In [ ]:
import pandas as pd

EFFECTIVENESS_10_COLUMNS = ['Recall@10', 'Precision@10', 'MRR@10', 'Hit@10', 'NDCG@10']
EFFECTIVENESS_20_COLUMNS = ['Recall@20', 'Precision@20', 'MRR@20', 'Hit@20', 'NDCG@20']
EFFICIENCY_COLUMNS = ['Epochs', 'Train', 'Eval', 'Params', 'FLOPs', 'CPU', 'GPU', 'MEM']


######################################################

def effectiveness_table(data_cols, table_name):
    # =========================
    # CONFIGURATION
    # =========================
    ID_COLUMNS = ['Dataset', 'Datatype', 'Model']
    COLUMNS_TO_USE = ID_COLUMNS + data_cols
    
    CSV_PATH = "data/result/df.csv"
    OUTPUT_TEX = f"data/result/{table_name}.tex"
    CSV_SEPARATOR = "\t"

    MODEL_ORDER = ["ItemKNN", "BPR", "NeuMF", "GCMC"]
    DATASET_ORDER = ["amazon_all_beauty", "ml-100k", "douban", "jester"]
    DATATYPE_ORDER = ["expR", "impR", "impB"]

    # Formatting
    FLOAT_FMT = "{:.4f}"
    EPS = 1e-12

    # Row rules (data rows only)
    DOTTED_EVERY = 4
    DASHED_EVERY = 12

    # =========================
    # HELPERS
    # =========================
    def escape_underscores(s: str) -> str:
        return s.replace("_", r"\_")

    def is_wrapped(s: str, cmd: str) -> bool:
        s = str(s).strip()
        return s.startswith(f"\\{cmd}{{") and s.endswith("}")

    def join_ints(xs):
        return ",".join(str(x) for x in xs)

    # =========================
    # LOAD DATA
    # =========================
    df = pd.read_csv(CSV_PATH, sep=CSV_SEPARATOR)
    df = df[COLUMNS_TO_USE].copy()

    metric_cols = [c for c in df.columns if c not in ID_COLUMNS]

    # Make metrics numeric
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Escape underscores in ID columns
    for c in ID_COLUMNS:
        df[c] = df[c].astype(str).map(escape_underscores)

    # Escape underscores in headers too
    escaped_headers = [escape_underscores(c) for c in df.columns]

    # =========================
    # ORDERING
    # =========================
    MODEL_ORDER = [escape_underscores(x) for x in MODEL_ORDER]
    DATASET_ORDER = [escape_underscores(x) for x in DATASET_ORDER]
    DATATYPE_ORDER = [escape_underscores(x) for x in DATATYPE_ORDER]

    df["Model"] = pd.Categorical(df["Model"], categories=MODEL_ORDER, ordered=True)
    df["Dataset"] = pd.Categorical(df["Dataset"], categories=DATASET_ORDER, ordered=True)
    df["Datatype"] = pd.Categorical(df["Datatype"], categories=DATATYPE_ORDER, ordered=True)

    # Dataset -> Datatype -> Model
    df = df.sort_values(["Dataset", "Datatype", "Model"]).reset_index(drop=True)

    # Numeric copy for aggregation logic
    df_num = df.copy()
    for c in metric_cols:
        df_num[c] = pd.to_numeric(df_num[c], errors="coerce")

    # IMPORTANT: convert ID columns to plain strings before injecting LaTeX macros
    for c in ID_COLUMNS:
        df[c] = df[c].astype(str)

    # =========================
    # HIGHLIGHT VALUES (ORIGINAL): bold best model per (dataset, datatype) per metric
    # =========================
    for c in metric_cols:
        best_model = df_num.groupby(["Dataset", "Datatype"], observed=True)[c].transform("max")

        formatted = []
        for x, bm in zip(df_num[c].tolist(), best_model.tolist()):
            if pd.isna(x):
                formatted.append("")
                continue

            val = float(x)
            s = FLOAT_FMT.format(val)

            is_bold = (not pd.isna(bm)) and abs(val - float(bm)) <= EPS
            if is_bold:
                s = f"\\textbf{{{s}}}"

            formatted.append(s)

        df[c] = formatted

    # =========================
    # RANK-BASED SUMMARY HIGHLIGHTS (scale-invariant)
    #   - Bold MODEL name: best per dataset (averaged over datatypes; mean rank over metrics)
    #   - Underline DATATYPE: best per (dataset, model) (mean rank over metrics)
    # =========================

    # --- 1) Bold MODEL name: best model per DATASET, averaged over DATATYPES
    # Rank MODELS within each (dataset, datatype) for each metric (best = rank 1)
    model_ranks = (
        df_num
        .groupby(["Dataset", "Datatype"], observed=True, dropna=False)[metric_cols]
        .rank(ascending=False, method="average")
    )

    # Mean rank over metrics -> one score per row (dataset, datatype, model). Lower is better.
    row_model_score = model_ranks.mean(axis=1)

    # Average over datatypes -> one score per (dataset, model)
    ds_model_score = row_model_score.groupby([df_num["Dataset"], df_num["Model"]], observed=True).mean()

    # Pick best model per dataset (min score)
    best_score = ds_model_score.groupby(level=0, observed=True).transform("min")
    best_models = ds_model_score.loc[(ds_model_score - best_score).abs() <= EPS]
    best_model_set = set(best_models.index.tolist())  # {(dataset, model), ...}

    # Bold the model name for all rows belonging to the winning (dataset, model)
    for i in range(len(df)):
        key = (df_num.at[i, "Dataset"], df_num.at[i, "Model"])
        if key in best_model_set:
            m = df.at[i, "Model"]
            if m and not is_wrapped(m, "textbf"):
                df.at[i, "Model"] = f"\\textbf{{{m}}}"


    # --- Underline DATATYPE: best datatype per MODEL (averaged over all DATASETS), rank-based

    # Rank DATATYPES within each (dataset, model) for each metric (best = rank 1)
    dt_ranks = (
        df_num
        .groupby(["Dataset", "Model"], observed=True, dropna=False)[metric_cols]
        .rank(ascending=False, method="average")
    )

    # Mean rank over metrics per row (dataset, datatype, model). Lower is better.
    row_dt_score = dt_ranks.mean(axis=1)

    # Average over datasets -> score per (model, datatype)
    model_dt_score = row_dt_score.groupby([df_num["Model"], df_num["Datatype"]], observed=True).mean()

    # Pick best datatype per model (min score)
    best_score = model_dt_score.groupby(level=0, observed=True).transform("min")
    best_model_dts = model_dt_score.loc[(model_dt_score - best_score).abs() <= EPS]
    best_model_dt_set = set(best_model_dts.index.tolist())  # {(model, datatype), ...}

    # Underline datatype everywhere it appears for that model (across all datasets)
    for i in range(len(df)):
        key = (df_num.at[i, "Model"], df_num.at[i, "Datatype"])
        if key in best_model_dt_set:
            d = df.at[i, "Datatype"]
            if d and not is_wrapped(d, "underline"):
                df.at[i, "Datatype"] = f"\\underline{{{d}}}"
                

    # =========================
    # BUILD TABULARRAY (tblr) LATEX
    # =========================
    n_cols = len(df.columns)
    n_data = len(df)

    colspec = ("l" * len(ID_COLUMNS)) + ("r" * (n_cols - len(ID_COLUMNS)))

    dotted_hlines, dashed_hlines = [], []
    for k in range(1, n_data + 1):  # data row index (1-based)
        hline_index = k + 2         # rule after data row k
        if k % DASHED_EVERY == 0:
            dashed_hlines.append(hline_index)
        elif k % DOTTED_EVERY == 0:
            dotted_hlines.append(hline_index)

    lines_opts = []
    if dotted_hlines:
        lines_opts.append(f"hline{{{join_ints(dotted_hlines)}}} = {{dashed}}")
    if dashed_hlines:
        idxs = join_ints(dashed_hlines)
        lines_opts.append(f"hline{{{idxs}}} = {{1}}{{-}}{{solid}}")
        lines_opts.append(f"hline{{{idxs}}} = {{2}}{{-}}{{solid}}")

    tblr_opts = ",\n  ".join([
        f"colspec={{ {colspec} }}",
        "row{1}={font=\\bfseries\\large}",
        f"cell{{1}}{{1-{n_cols}}}={{halign=l}}",
        f"cell{{2-Z}}{{1-{len(ID_COLUMNS)}}}={{halign=l}}",
        f"cell{{2-Z}}{{{len(ID_COLUMNS)+1}-{n_cols}}}={{halign=r}}",

        "hline{1} = {1}{-}{solid}",
        "hline{1} = {2}{-}{solid}",
        "hline{2} = {1}{-}{solid}",
        "hline{2} = {2}{-}{solid}",
        *lines_opts,
        "hline{Z} = {1}{-}{solid}",
        "hline{Z} = {2}{-}{solid}",
    ])

    header_row = " & ".join(escaped_headers) + r" \\"

    body_rows = []
    for _, row in df.iterrows():
        body_rows.append(" & ".join(map(str, row.tolist())) + r" \\")

    latex_lines = []
    latex_lines.append("% Auto-generated by Python (pandas) + tabularray")
    latex_lines.append("% Requires in LaTeX preamble:")
    latex_lines.append("% \\usepackage{tabularray}")
    latex_lines.append("% \\UseTblrLibrary{booktabs}")
    latex_lines.append("")
    latex_lines.append("\\begin{tblr}[]{")
    latex_lines.append("  " + tblr_opts)
    latex_lines.append("}")
    latex_lines.append(header_row)
    latex_lines.extend(body_rows)
    latex_lines.append("\\end{tblr}")
    latex_lines.append("")

    with open(OUTPUT_TEX, "w", encoding="utf-8") as f:
        f.write("\n".join(latex_lines))

    print(f"Saved LaTeX table to {OUTPUT_TEX}")

#######################################################

def efficiency_table(data_cols=EFFICIENCY_COLUMNS, table_name="efficiency"):
    # =========================
    # CONFIGURATION
    # =========================
    ID_COLUMNS = ['Dataset', 'Datatype', 'Model']
    COLUMNS_TO_USE = ID_COLUMNS + data_cols

    CSV_PATH = "data/result/df.csv"
    OUTPUT_TEX = f"data/result/{table_name}.tex"
    CSV_SEPARATOR = "\t"

    MODEL_ORDER = ["ItemKNN", "BPR", "NeuMF", "GCMC"]
    DATASET_ORDER = ["amazon_all_beauty", "ml-100k", "douban", "jester"]
    DATATYPE_ORDER = ["expR", "impR", "impB"]

    # Column typing (per your description)
    STRING_COLS = set(ID_COLUMNS + [])
    INT_COLS = {'Epochs', 'FLOPs', 'Params'}
    FLOAT_COLS = {'Train', 'Eval', 'CPU', 'GPU', 'MEM'}

    # Row rules (data rows only)
    DOTTED_EVERY = 4
    DASHED_EVERY = 12

    # =========================
    # HELPERS
    # =========================
    def latex_escape(s: str) -> str:
        s = str(s)
        return (s.replace("\\", r"\textbackslash{}")
                .replace("&", r"\&")
                .replace("%", r"\%")
                .replace("$", r"\$")
                .replace("#", r"\#")
                .replace("_", r"\_")
                .replace("{", r"\{")
                .replace("}", r"\}")
                .replace("~", r"\textasciitilde{}")
                .replace("^", r"\textasciicircum{}"))

    def join_ints(xs):
        return ",".join(str(x) for x in xs)

    def fmt_cell(col, val):
        # empty for missing
        if pd.isna(val):
            return ""
        if col in FLOAT_COLS:
            return f"{float(val):.2f}"
        if col in INT_COLS:
            # Int64 can be numpy/pandas scalar; cast safely
            return str(int(val))
        # strings (also escape underscores for LaTeX)
        return latex_escape(val)

    # =========================
    # LOAD DATA
    # =========================
    df = pd.read_csv(CSV_PATH, sep=CSV_SEPARATOR)
    df = df[COLUMNS_TO_USE].copy()

    # =========================
    # TYPES (fix)
    # =========================
    # Strings first (keep as strings; escape underscores later via fmt_cell)
    for c in STRING_COLS:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Ints
    for c in INT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # Floats
    for c in FLOAT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)

    # =========================
    # ORDERING
    # =========================
    # (ordering uses raw values; escaping happens at render time)
    df["Model"] = pd.Categorical(df["Model"], categories=MODEL_ORDER, ordered=True)
    df["Dataset"] = pd.Categorical(df["Dataset"], categories=DATASET_ORDER, ordered=True)
    df["Datatype"] = pd.Categorical(df["Datatype"], categories=DATATYPE_ORDER, ordered=True)

    df = df.sort_values(["Dataset", "Datatype", "Model"]).reset_index(drop=True)

    # Escape underscores in headers
    escaped_headers = [latex_escape(c) for c in df.columns]

    # =========================
    # BUILD TABULARRAY (tblr) LATEX
    # =========================
    n_cols = len(df.columns)
    n_data = len(df)

    colspec = ("l" * len(ID_COLUMNS)) + ("r" * (n_cols - len(ID_COLUMNS)))

    dotted_hlines, dashed_hlines = [], []
    for k in range(1, n_data + 1):  # data row index (1-based)
        hline_index = k + 2         # rule after data row k
        if k % DASHED_EVERY == 0:
            dashed_hlines.append(hline_index)
        elif k % DOTTED_EVERY == 0:
            dotted_hlines.append(hline_index)

    lines_opts = []
    if dotted_hlines:
        lines_opts.append(f"hline{{{join_ints(dotted_hlines)}}} = {{dashed}}")
    if dashed_hlines:
        idxs = join_ints(dashed_hlines)
        lines_opts.append(f"hline{{{idxs}}} = {{1}}{{-}}{{solid}}")
        lines_opts.append(f"hline{{{idxs}}} = {{2}}{{-}}{{solid}}")

    tblr_opts = ",\n  ".join([
        f"colspec={{ {colspec} }}",
        "row{1}={font=\\bfseries\\large}",
        f"cell{{1}}{{1-{n_cols}}}={{halign=l}}",
        f"cell{{2-Z}}{{1-{len(ID_COLUMNS)}}}={{halign=l}}",
        f"cell{{2-Z}}{{{len(ID_COLUMNS)+1}-{n_cols}}}={{halign=r}}",
        "hline{1} = {1}{-}{solid}",
        "hline{1} = {2}{-}{solid}",
        "hline{2} = {1}{-}{solid}",
        "hline{2} = {2}{-}{solid}",
        *lines_opts,
        "hline{Z} = {1}{-}{solid}",
        "hline{Z} = {2}{-}{solid}",
    ])

    header_row = " & ".join(escaped_headers) + r" \\"

    body_rows = []
    cols = list(df.columns)
    for _, row in df.iterrows():
        formatted = [fmt_cell(col, row[col]) for col in cols]
        body_rows.append(" & ".join(formatted) + r" \\")

    latex_lines = []
    latex_lines.append("% Auto-generated by Python (pandas) + tabularray")
    latex_lines.append("% Requires in LaTeX preamble:")
    latex_lines.append("% \\usepackage{tabularray}")
    latex_lines.append("% \\UseTblrLibrary{booktabs}")
    latex_lines.append("")
    latex_lines.append("\\begin{tblr}[]{")
    latex_lines.append("  " + tblr_opts)
    latex_lines.append("}")
    latex_lines.append(header_row)
    latex_lines.extend(body_rows)
    latex_lines.append("\\end{tblr}")
    latex_lines.append("")

    with open(OUTPUT_TEX, "w", encoding="utf-8") as f:
        f.write("\n".join(latex_lines))

    print(f"Saved LaTeX table to {OUTPUT_TEX}")

effectiveness_table(EFFECTIVENESS_10_COLUMNS, "effectiveness@10")
effectiveness_table(EFFECTIVENESS_20_COLUMNS, "effectiveness@20")
efficiency_table()